# Aliasing demo

This notebook demonstrates the aliasing capabilities of the ACCESS-NRI Intake catalog.
Aliasing lets you search the catalog using familiar CMIP-style vocabulary (field names,
variable codes, frequency strings, etc.) and have those terms automatically mapped to
the underlying canonical values stored in the catalog.

For a full reference, see the [Aliasing](aliases.rst) documentation page.

> **Note**: This notebook must be run on Gadi with access to the `xp65` project
> and the relevant data projects. See the [How do I use it?](how.rst) page for prerequisites.

## Contents

1. [Setup](#setup)
2. [Field aliasing — CMIP column names](#field-aliasing)
3. [Value aliasing — frequency](#value-aliasing-frequency)
4. [Value aliasing — realm](#value-aliasing-realm)
5. [Value aliasing — model shortcuts](#value-aliasing-model)
6. [Value aliasing — experiment shortcuts](#value-aliasing-experiment)
7. [CMIP variable search (scalar)](#cmip-variable-scalar)
8. [CMIP variable search (list)](#cmip-variable-list)
9. [Regex passthrough](#regex-passthrough)
10. [Controlling alias warnings](#warnings)
11. [Escaping aliasing with .unwrap()](#unwrap)

<a name="setup"></a>
## 1. Setup

Load the catalog. On Gadi this uses the default catalog location in `/g/data/xp65`.

In [ ]:
import warnings
import intake

cat = intake.cat.access_nri
cat

<a name="field-aliasing"></a>
## 2. Field aliasing — CMIP column names

The top-level catalog accepts CMIP-style column names in `.search()`. For example,
you can use `source_id` instead of `model`, or `variable_id` instead of `variable`.

Each call will emit a `UserWarning` showing the mapping that was applied.

In [ ]:
# Using CMIP-style 'source_id' — maps to the catalog's 'model' column
results = cat.search(source_id="ACCESS-ESM1-5")
results

In [ ]:
# Using 'variable_id' — maps to the catalog's 'variable' column
results = cat.search(variable_id="tas")
results

<a name="value-aliasing-frequency"></a>
## 3. Value aliasing — frequency

Human-readable frequency strings are mapped to the catalog's stored values
(e.g. `"daily"` → `"1day"`). The search expands to include **both** terms, so
any data already labelled `"daily"` is also included.

In [ ]:
# 'daily' expands to ['1day', 'daily']
results = cat.search(frequency="daily")
results

In [ ]:
# 'monthly' expands to ['1mon', 'monthly']
results = cat.search(frequency="monthly")
results

<a name="value-aliasing-realm"></a>
## 4. Value aliasing — realm

Common realm strings map to the catalog's stored values
(e.g. `"atmosphere"` → `"atmos"`, `"sea-ice"` → `"seaIce"`).

In [ ]:
# 'atmosphere' expands to ['atmos', 'atmosphere']
results = cat.search(realm="atmosphere")
results

In [ ]:
# 'sea-ice' expands to ['seaIce', 'sea-ice']
results = cat.search(realm="sea-ice")
results

<a name="value-aliasing-model"></a>
## 5. Value aliasing — model shortcuts

Short model names map to their full catalog names.

In [ ]:
# 'ACCESS-ESM1' expands to ['ACCESS-ESM1-5', 'ACCESS-ESM1']
results = cat.search(model="ACCESS-ESM1")
results

<a name="value-aliasing-experiment"></a>
## 6. Value aliasing — experiment shortcuts

Common experiment abbreviations and CMIP5 scenario names are mapped forward.

In [ ]:
# 'hist' expands to ['historical', 'hist']
results = cat.search(experiment_id="hist")
results

In [ ]:
# 'rcp85' expands to ['ssp585', 'rcp85']
results = cat.search(experiment_id="rcp85")
results

<a name="cmip-variable-scalar"></a>
## 7. CMIP variable search — scalar

CMIP standard variable names (e.g. `tas`, `pr`, `ci`) are mapped to the raw
ACCESS field codes stored in ESM datastores. The search always includes *both*
the CMIP name and the ACCESS code, so you never miss data labelled either way.

These mappings come from `access-esm1-6-cmip-mappings.json` which covers 137
atmosphere, land, and ocean variables for ACCESS-ESM1.6.

In [ ]:
# Get an ESM datastore for ACCESS-ESM1-6
# TODO: replace 'access-esm1-6' with the correct catalog entry name
ds = cat["access-esm1-6"]
ds

In [ ]:
# Search for 'tas' — CMIP name for near-surface air temperature
# Automatically expands to variable=['fld_s03i236', 'tas']
results = ds.search(variable="tas")
results

In [ ]:
# 'ci' (convection time fraction) → ['fld_s05i269', 'ci']
results = ds.search(variable="ci")
results

<a name="cmip-variable-list"></a>
## 8. CMIP variable search — list

You can pass a list of CMIP variable names. Each is aliased individually and
the full expanded set is searched.

In [ ]:
# Each CMIP name is resolved: ci→fld_s05i269, cl→fld_s02i261, tas→fld_s03i236
results = ds.search(variable=["ci", "cl", "tas"])
results

<a name="regex-passthrough"></a>
## 9. Regex passthrough

Regex patterns and other non-string values bypass aliasing entirely and are
passed directly to the underlying catalog. This allows you to use the full power
of intake-esm's regex search when you need it.

In [ ]:
# Regex string — passed through unchanged, no aliasing applied
results = ds.search(variable="ci|cl|tas")
results

<a name="warnings"></a>
## 10. Controlling alias warnings

Each alias expansion emits a `UserWarning`. You can suppress these using
Python's standard `warnings` module.

In [ ]:
# Suppress UserWarnings from aliasing
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    results = cat.search(frequency="daily", realm="atmosphere")

results

<a name="unwrap"></a>
## 11. Escaping aliasing with `.unwrap()`

Both the top-level catalog wrapper and the per-dataset wrapper expose an
`.unwrap()` method that returns the raw underlying catalog object with no
aliasing layer. Use this when you need the exact type expected by another
library or want to call a method not supported by the wrapper.

In [ ]:
# Get the raw DfFileCatalog (no aliasing wrapper)
raw_cat = cat.unwrap()
type(raw_cat)

In [ ]:
# Get the raw esm_datastore (no aliasing wrapper)
raw_ds = ds.unwrap()
type(raw_ds)